# 4. Ensembles de Árboles de Decisión
## 4.06 GBDT LightGBM (Ejecución Local)

La técnica de Gradient Boosting fue creada por Jerome H. Friedman en 1999 - 2001.
<br>Se implementaron librerías ineficientes al inicio; en 2016 se crea XGBoost y en 2017 LightGBM.

**Paper original de Gradient Boosting:**
* Friedman JH. *Greedy function approximation: A gradient boosting machine.* Ann Stat. 2001;29(5):1189–232. [Ver enlace](https://projecteuclid.org/journals/annals-of-statistics/volume-29/issue-5/Greedy-function-approximation-A-gradient-boosting-machine/10.1214/aos/1013203451.pdf)

**Paper XGBoost:**
* Chen, T.; Guestrin, C. *Xgboost: A scalable tree boosting system.* KDD 2016. [Ver enlace](https://dl.acm.org/doi/pdf/10.1145/2939672.2939785)

**Paper LightGBM:**
* Ke G. et al. *Lightgbm: A highly efficient gradient boosting decision tree.* NeurIPS 2017. [Ver enlace](https://proceedings.neurips.cc/paper/2017/file/6449f44a102fde848669bdd9eb6b76fa-Paper.pdf)

El Gradient Boosting of Decision Trees es un ensemble de árboles de decisión donde la construcción es secuencial: cada nuevo árbol se genera para predecir el error/gradiente del ensemble acumulado hasta ese punto.

Cada árbol de LightGBM se entrena sobre un dataset perturbado (utilizando un porcentaje `feature_fraction` de los atributos al azar).

#### 4.06.1 Seteo del ambiente local
Configuración de directorios de trabajo y verificación/descarga del dataset.

In [1]:
# Determinar la raíz del proyecto de forma absoluta
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
setwd(dir_base)

# Configurar entorno para proxy y herramientas CLI (Kaggle)
Sys.setenv(
  http_proxy = "http://10.4.8.20:8080",
  https_proxy = "http://10.4.8.20:8080",
  HTTP_PROXY = "http://10.4.8.20:8080",
  HTTPS_PROXY = "http://10.4.8.20:8080",
  PATH = paste("/home/sectorial/anaconda3/bin", file.path(Sys.getenv("HOME"), ".local/bin"), Sys.getenv("PATH"), sep = ":")
)

# Crear carpetas locales para datasets y experimentos si no existen
dir.create(file.path(dir_base, "datasets"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(dir_base, "exp"), showWarnings = FALSE, recursive = TRUE)

# Descargar el dataset si no existe localmente
url_dataset <- "https://storage.googleapis.com/open-courses/utn2026-b40a/dataset_pequeno.csv"
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")

if (!file.exists(archivo_dataset)) {
  cat("Descargando dataset_pequeno.csv...\n")
  download.file(url_dataset, destfile = archivo_dataset, mode = "wb")
  cat("Descarga completada exitosamente.\n")
} else {
  cat("El dataset ya se encuentra disponible en:", archivo_dataset, "\n")
}

El dataset ya se encuentra disponible en: /workspace/datasets/dataset_pequeno.csv 


### 4.07 LightGBM, una corrida
#### 4.07.1 Inicio y limpieza del ambiente

In [2]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Aug 27 17:15:34 2026"

In [3]:
# limpio la memoria
rm(list = ls(all.names = TRUE)) # remove all objects
gc(full = TRUE, verbose = FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,661718,35.4,1454511,77.7,1454511,77.7
Vcells,1232242,9.5,8388608,64.0,1975054,15.1


#### 4.07.2 Carga de librerías

In [4]:
# cargo las librerias que necesito
if (!require("data.table")) install.packages("data.table", repos = "https://cloud.r-project.org")
require("data.table")

if (!require("rpart")) install.packages("rpart", repos = "https://cloud.r-project.org")
require("rpart")

if (!require("rlist")) install.packages("rlist", repos = "https://cloud.r-project.org")
require("rlist")

if (!require("lightgbm")) install.packages("lightgbm", repos = "https://cloud.r-project.org")
require("lightgbm")

Loading required package: data.table

Loading required package: rpart

Loading required package: rlist

Loading required package: lightgbm



#### 4.07.3 Definición de Parámetros e Hiperparámetros

In [5]:
PARAM <- list()
PARAM$experimento <- '4070_01'
PARAM$semilla_primigenia <- 115879 # Reemplace con su semilla primigenia

# estos hiperparametros de LightGBM surgieron de una Bayesian Optimization
# Parámetros de la corrida original
# PARAM$lgb$num_iterations <- 1000  # cantidad de arbolitos
# PARAM$lgb$learning_rate <- 0.027
# PARAM$lgb$feature_fraction <- 0.8
# PARAM$lgb$min_data_in_leaf <- 76
# PARAM$lgb$num_leaves <- 8
# PARAM$lgb$max_bin <- 31

# Parámetros de corrida local
PARAM$lgb$num_iterations <- 600  # cantidad de arbolitos
PARAM$lgb$learning_rate <- 0.02
PARAM$lgb$feature_fraction <- 0.4
PARAM$lgb$min_data_in_leaf <- 200
PARAM$lgb$num_leaves <- 120
PARAM$lgb$max_bin <- 31

#### 4.07.4 Carpeta de trabajo y lectura de datos

In [6]:
# Carpeta de trabajo local para el experimento
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
experimento_folder <- paste0("KA", PARAM$experimento)
dir_exp <- file.path(dir_base, "exp", experimento_folder)
dir.create(dir_exp, showWarnings = FALSE, recursive = TRUE)
setwd(dir_exp)
cat("Directorio de trabajo actual:", getwd(), "\n")

Directorio de trabajo actual: /workspace/exp/KA4070_01 


In [7]:
# Lectura del dataset desde la carpeta local
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")
dataset <- fread(archivo_dataset, stringsAsFactors = TRUE)

# Paso la clase a binaria (en este script base solo BAJA+2 es 1L)
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+2"), 1L, 0L)]

# Campos que se van a utilizar
campos_buenos <- setdiff(colnames(dataset), c("clase_ternaria", "clase01"))

# Establezco dónde entreno (202107)
dataset[, train := 0L]
dataset[foto_mes %in% c(202107), train := 1L]

# Creo la estructura Dataset propia de LightGBM
dtrain <- lgb.Dataset(
  data = data.matrix(dataset[train == 1L, campos_buenos, with = FALSE]),
  label = dataset[train == 1L, clase01]
)

cat("Registros en train:", nrow(dtrain), "| Variables:", ncol(dtrain), "\n")

Registros en train: 164479 | Variables: 154 


#### 4.07.5 Entrenamiento del Modelo LightGBM

In [8]:
# Establezco la semilla aleatoria
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")

# Entreno el modelo
modelo <- lgb.train(
  data = dtrain,
  param = list(
    objective = "binary",
    max_bin = PARAM$lgb$max_bin,
    learning_rate = PARAM$lgb$learning_rate,
    num_iterations = PARAM$lgb$num_iterations,
    num_leaves = PARAM$lgb$num_leaves,
    min_data_in_leaf = PARAM$lgb$min_data_in_leaf,
    feature_fraction = PARAM$lgb$feature_fraction,
    seed = PARAM$semilla_primigenia
  )
)

cat("Entrenamiento finalizado exitosamente.\n")

[LightGBM] [Info] Number of positive: 1304, number of negative: 163175
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.093748 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3565
[LightGBM] [Info] Number of data points in the train set: 164479, number of used features: 150
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007928 -> initscore=-4.829387
[LightGBM] [Info] Start training from score -4.829387
Entrenamiento finalizado exitosamente.


In [9]:
# Imprimo y grabo la importancia de variables
tb_importancia <- as.data.table(lgb.importance(modelo))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file = archivo_importancia,
  sep = "\t"
)

cat("Top 10 variables más importantes:\n")
print(head(tb_importancia, 10))

# Grabo a disco el modelo en formato texto
lgb.save(modelo, "modelo.txt")

Top 10 variables más importantes:
                 Feature       Gain      Cover   Frequency
                  <char>      <num>      <num>       <num>
 1:         ctrx_quarter 0.09140023 0.02033017 0.017773109
 2:         mcaja_ahorro 0.04773741 0.01017152 0.016610644
 3:    mcuenta_corriente 0.03027183 0.01858215 0.024327731
 4:      mpasivos_margen 0.02837014 0.01264538 0.023277311
 5:     Visa_mpagospesos 0.02678084 0.01187116 0.019873950
 6:       mcuentas_saldo 0.02604458 0.01146549 0.022829132
 7:         cpayroll_trx 0.02443958 0.01240076 0.002591036
 8:             mpayroll 0.02443227 0.02469161 0.004873950
 9:         cliente_edad 0.02436760 0.02222020 0.032997199
10: mrentabilidad_annual 0.02284192 0.02073028 0.027310924


#### 4.07.6 Predicción y Generación de Envío para Kaggle

In [10]:
# Aplico el modelo a los datos del futuro (202109)
dfuture <- dataset[foto_mes == 202109]

prediccion <- predict(
  modelo,
  data.matrix(dfuture[, campos_buenos, with = FALSE])
)

# Tabla de predicción
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# Grabo las probabilidades del modelo
fwrite(tb_prediccion,
  file = "prediccion.txt",
  sep = "\t"
)
cat("Predicciones guardadas en prediccion.txt\n")

Predicciones guardadas en prediccion.txt


In [11]:
# Ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

# Genero la predicción binaria con umbral 1/40 = 0.025
tb_prediccion[, Predicted := 0L]
tb_prediccion[prob > (1 / 40), Predicted := 1L]

archivo_kaggle <- paste0("KA", PARAM$experimento, ".csv")

# Grabo el archivo para Kaggle
fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
  file = archivo_kaggle,
  sep = ","
)

cat("Archivo generado:", archivo_kaggle, "con", tb_prediccion[Predicted == 1L, .N], "clientes positivos.\n")

# Subida a Kaggle vía CLI
kaggle_bin <- Sys.which("kaggle")
if (kaggle_bin == "") kaggle_bin <- "/home/sectorial/anaconda3/bin/kaggle"

mensaje <- paste0("num_iterations=", PARAM$lgb$num_iterations,
  " learning_rate=", PARAM$lgb$learning_rate,
  " feature_fraction=", PARAM$lgb$feature_fraction,
  " min_data_in_leaf=", PARAM$lgb$min_data_in_leaf,
  " num_leaves=", PARAM$lgb$num_leaves,
  " max_bin=", PARAM$lgb$max_bin
)

linea <- paste0(
  kaggle_bin, " competitions submit -c utn-2026-inicial",
  " -f ", archivo_kaggle,
  " -m '", mensaje, "'"
)

tryCatch({
  salida <- system(linea, intern = TRUE)
  cat(paste(salida, collapse = "\n"), "\n")
}, error = function(e) {
  cat("Nota: No se pudo realizar el submit automático:", conditionMessage(e), "\n")
  cat("Puede subir manualmente el archivo:", archivo_kaggle, "\n")
})

format(Sys.time(), "%a %b %d %X %Y")

Archivo generado: KA4070_01.csv con 5081 clientes positivos.
Nota: No se pudo realizar el submit automático: error in running command: 'Function not implemented' 
Puede subir manualmente el archivo: KA4070_01.csv 


[1] "Thu Aug 27 17:18:27 2026"